In [1]:
using KernelFunctions
using AbstractGPs
using Serialization
using Turing
using LinearAlgebra
using Statistics
using Optim
using Serialization 
using MCMCChains
using ForwardDiff
using BayesSoundSource
using PythonPlot 


SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up


In [2]:
chn = deserialize("chn_joint.bin")

# @assert all(rhat -> isapprox(rhat, 1; atol=0.05), rhat(chn).nt.rhat) "Deserialised chain may not have converged"

N = 100
dims = 3
Z = (size(group(chn, "traj.coords"), 2) ÷ dims)

subchn = sample(chn, N) 

ts = eachrow(Array(group(subchn, :t)))
t_test = range(extrema(Iterators.flatten(ts))..., 300)
# t_test = range(0, 5, 300)


traj = sample_traj(subchn, 3, Z)

ells = vec(subchn["traj.ℓ"])
sigma_noises = vec(subchn["traj.σ_noise"])
sigma_rbfs = vec(subchn["traj.σ_rbf"])
;

In [ ]:

jitter = 0.000001

distance = []
velocity = []
azimuth = []
elevation =[]
acceleration = []
paths = []


for i in 1:N
    x = traj[:, 1, i]
    y = traj[:, 2, i]
    z = traj[:, 3, i]
    t = ts[i]
    ℓ = ells[i]
    σ_noise = sigma_noises[i]
    σ_rbf = sigma_rbfs[i]
    kern = σ_rbf * with_lengthscale(SEKernel(), ℓ) 

### Distance 

    K = kern.(t, t') + σ_noise * I
    K_s = kern.(t, t_test')
    K_ss = kern.(t_test, t_test')

    μ_x = K_s' * inv(K) * x
    μ_y = K_s' * inv(K) * y
    μ_z = K_s' * inv(K) * z
    Σ = Symmetric(K_ss - K_s' * inv(K) * K_s)

    xs = rand(MvNormal(μ_x, Σ + jitter*I), 10)
    ys = rand(MvNormal(μ_y, Σ + jitter*I), 10)
    zs = rand(MvNormal(μ_z, Σ + jitter*I), 10)

    push!(distance, eachcol(map(norm, zip(xs,ys,zs)))...)
    push!(paths, (μ_x, μ_y, μ_z))


### Velocity 

    k_f_df(x, x′) = ForwardDiff.derivative(s -> kern(x, s), x′)
    k_df_df(x, x′) = ForwardDiff.derivative(
        t -> ForwardDiff.derivative(
            s -> kern(t, s), x′
        ),
        x
    )

    K = kern.(t, t') + σ_noise * I
    K_s = k_f_df.(t, t_test')
    K_ss = k_df_df.(t_test, t_test')

    # μ_df = K_s' * inv(K) * z
    Σ_df = K_ss - K_s' * inv(K) * K_s

    μ_x = K_s' * inv(K) * x
    μ_y = K_s' * inv(K) * y
    μ_z = K_s' * inv(K) * z
    Σ = Symmetric(K_ss - K_s' * inv(K) * K_s)

    xs = rand(MvNormal(μ_x, Σ + jitter*I), 20)
    ys = rand(MvNormal(μ_y, Σ + jitter*I), 20)
    zs = rand(MvNormal(μ_z, Σ + jitter*I), 20)

    push!(velocity, eachcol(map(norm, zip(xs,ys,zs)))...)
    push!(azimuth, eachcol(atan.(xs,ys))...)
    push!(elevation, eachcol(@. atan(zs, sqrt(xs^2 + ys^2)))...)

### Accelaration
    k_f_d2f(x, y) = 
    ForwardDiff.derivative(x) do x′
        ForwardDiff.derivative(y) do y′
            kern(x′, y′)
        end 
    end 

    k_d2f_d2f(x, y) = 
    ForwardDiff.derivative(x) do x′
        ForwardDiff.derivative(x′) do x′′ 
            ForwardDiff.derivative(y) do y′
                ForwardDiff.derivative(y′) do y′′
                    kern(x′′, y′′)
                end 
            end 
        end 
    end 

    K = kern.(t, t') + σ_noise * I
    K_s = k_f_d2f.(t, t_test')
    K_ss = k_d2f_d2f.(t_test, t_test')

    μ_x = K_s' * inv(K) * x
    μ_y = K_s' * inv(K) * y
    μ_z = K_s' * inv(K) * z
    Σ = Symmetric(K_ss - K_s' * inv(K) * K_s)

    xs = rand(MvNormal(μ_x, Σ + jitter*I), 20)
    ys = rand(MvNormal(μ_y, Σ + jitter*I), 20)
    zs = rand(MvNormal(μ_z, Σ + jitter*I), 20)

    push!(acceleration, eachcol(map(norm, zip(xs, ys, zs)))...)
end 


# Plotting 

In [ ]:
microphone_coords = [
    [0.0, 0.0, 0.0],
    [-8.0, 0.0, 0.89],
    [0.0, 8.0, 1.65],
    [8.0, 0.0, 0.92],
    [0.0, -8.0, 1.64]
]
fig = figure(figsize=(8, 6), constrained_layout=true)
ax = fig.add_subplot(111, projection="3d")
ax.set_proj_type("ortho") 
ax.set_zlim(0, 8)
ax.set_xlim(-10, 10)
ax.set_ylim(-15, 10)
ax.view_init(elev=20, azim=30)

ax.set_xticks(-10:5:10)
ax.set_yticks(-10:5:10)
ax.set_zticks(0:2:8)
ax.xaxis.pane.fill = false
ax.yaxis.pane.fill = false
ax.zaxis.pane.fill = false

ax.set_xlabel("X Offset (m)")
ax.set_ylabel("Y Offset (m)")
ax.set_zlabel("Height (m)")
ax.zaxis.labelpad=-1

ax.grid(true, alpha=0.2, linewidth=0.6)

colours = [
    "#4C72B0",
    "#55A868",
    "#C44E52",
    "#8172B3",
    "#CCB974",
    "#64B5CD"
]

for (xs, ys, zs) ∈ sample(paths, 30)
    ax.plot(
        xs,
        ys,
        zs;
        label="2D Project",
        color="black",
        alpha=0.6,
        linewidth=0.6
    )
end

# # # 2D projection

xs = mean.(eachrow(hcat(getindex.(paths, 1)...)))
ys = mean.(eachrow(hcat(getindex.(paths, 2)...)))
zs = mean.(eachrow(hcat(getindex.(paths, 3)...)))
ax.plot(
    xs,
    ys,
    zero(length(zs));
    label="2D Project",
    color="gray",
    linestyle="--",
    alpha=1,
    linewidth=1, 
    zorder=0
)

ax.plot(
    xs,
    zero(length(ys)) - 15,
    zs;
    label="2D Project",
    color="gray",
    linestyle="--",
    alpha=1,
    linewidth=1, 
    zorder=0
)

ax.plot(
    zero(length(xs)) - 10,
    ys,
    zs;
    label="2D Project",
    color="gray",
    linestyle="--",
    alpha=1,
    linewidth=1, 
    zorder=0
)


# # # Receiver locations 


# Receiver positions
mx = getindex.(microphone_coords, 1)
my = getindex.(microphone_coords, 2)
mz = getindex.(microphone_coords, 3)

ax.scatter(mx, my, mz; marker="x", label="Receivers",  s=30, linewidths=1.2, c=fill("black", 5), alpha=1)

# Vertical guide lines
ax.plot([-8.0, -8.0], [0.0, 0.0], [0.89, 0.0], c="black")
ax.plot([0.0, 0.0], [8.0, 8.0], [1.65, 0.0], c="black")
ax.plot([8.0, 8.0], [0.0, 0.0], [0.92, 0.0], c="black")
ax.plot([0.0, 0.0], [-8.0, -8.0], [1.64, 0.0], c="black")

# ax.set_position([0, 0.5, 0.9, 0.9])
fig

In [ ]:
function gp_plot(mu, stddev, samples, t_train, t_test, y_label) 

    colours = [ "#4C72B0", "#55A868", "#C44E52", "#8172B3", "#CCB974", "#64B5CD"]
    fig, ax = subplots(figsize=(12, 4))
    subplots_adjust(left=0.07, right=0.95, bottom=0.15, top=0.95)

    ax.set_xlabel("Time (s)", fontsize=20)
    ax.set_ylabel(y_label, fontsize=20)
    # ax.set_xlim(0,3)
    # ax.set_ylim(0,10)

    ax.grid(
        true;
        which="minor",
        axis="x",
        linewidth=0.6,
        color="black",
        alpha=0.2,
        linestyle=":"
    )

    ax.set_xticks(t_train; minor=true)

    ax.tick_params(
        axis="x",
        which="minor",
        bottom=true,
        top=false,
        direction="in",
        length=4,
        width=0.8
    )
    # Mean line
    ax.plot(t_test, mu, color=colours[1])

    # 1-sigma band
    ax.fill_between(t_test, mu .- stddev, mu .+ stddev,
        color=colours[1], alpha=0.3)

    # 2-sigma band
    ax.fill_between(t_test, mu .- 2 .* stddev, mu .+ 2 .* stddev,
        color=colours[1], alpha=0.3)

    # Sample trajectories
    for s in samples
        ax.plot(t_test, s, color=colours[3], alpha=0.4, zorder=10)
    end
    return fig
end 

In [ ]:
mu = rad2deg.(mean.(eachrow(hcat(elevation...))))
stddev = rad2deg.(std.(eachrow(hcat(elevation...))))
samples = map(s -> rad2deg.(s), sample(elevation, 10))


t_train = mean(eachrow(Array(group(chn, :t))))

fig = gp_plot(mu, stddev, samples, t_train, t_test, "Pitch (degrees)") 

In [ ]:
mu = rad2deg.(mean.(eachrow(hcat(azimuth...))))
stddev = rad2deg.(std.(eachrow(hcat(azimuth...))))
samples = map(s -> rad2deg.(s), sample(azimuth, 10))

t_train = mean(eachrow(Array(group(chn, :t))))

fig = gp_plot(mu, stddev, samples, t_train, t_test, "Yaw (degrees)") 

In [ ]:
mu = mean.(eachrow(hcat(velocity...)))
stddev = std.(eachrow(hcat(velocity...)))
samples = sample(velocity, 10)

t_train = mean(eachrow(Array(group(chn, :t))))

fig = gp_plot(mu, stddev, samples, t_train, t_test, "Speed (m/s)") 

In [ ]:
mu = mean.(eachrow(hcat(acceleration...)))
stddev = std.(eachrow(hcat(acceleration...)))
samples = sample(acceleration, 10)

t_train = mean(eachrow(Array(group(chn, :t))))

fig = gp_plot(mu, stddev, samples, t_train, t_test, "Acceleration (m/s²)") 

In [ ]:

x = traj[:, 1, :]
y = traj[:, 2, :]
z = traj[:, 3, :]
t = transpose(Array(group(subchn, :t)))

δx = x[2:end,:] .- x[1:end-1,:]
δy = y[2:end,:] .- y[1:end-1,:]
δz = z[2:end,:] .- z[1:end-1,:]
δt = t[2:end,:] .- t[1:end-1,:]

samples = size(traj, 3)
Z = size(traj, 1) - 1
mag(dx,dy,dz, dt) = sqrt(dx^2 + dy^2 + dz^2) / dt
vel = map(1:samples) do i
    map(1:Z) do z
        mag(δx[z,i], δy[z,i], δz[z,i], δt[z,i])
    end 
end


mu = mean.(eachrow(hcat(vel...)))
stddev = std.(eachrow(hcat(vel...)))

t_train = mean(eachrow(Array(group(chn, :t))))

fig = gp_plot(mu, stddev, [], t_train, (t_train[1:end-1] .+ t_train[2:end]) ./ 2, "Speed (m/s)") 